# Backends: the array shim

`specdiff/ops.py` is the **only** module that touches NumPy or PyTorch. Everything else — the
samplers, the trees, the verifiers — works through a ~20-method `Backend` protocol, so porting the
library to JAX or MLX means writing one small subclass and nothing else.

Two conventions hold everywhere:

* **states are stacks**: shape `(num_nodes, *state_shape)`, where `state_shape` is whatever a
  single diffusion state is for your model — `(d,)`, `(3, 32, 32)`, `(16, 64, 64)` for an SD3
  latent;
* **indices are plain Python ints**: node ids, step indices, batch indices. Nothing
  framework-specific crosses the public API except the state arrays themselves.

In [1]:
import numpy as np
import torch

from specdiff import (
    DelayedDriftProposal,
    BatchedSpeculativeSampler,
    DraftTree,
    NoiseSchedule,
    SpeculativeSampler,
    TargetTransition,
    Verifier,
    VerifyResult,
    check_exactness,
    resolve_backend,
)
from specdiff.ops import Backend, NumpyBackend, TorchBackend, standard_normal_cdf, standard_normal_sf
from specdiff.verifiers.rank1 import Rank1Frame

## 1. Resolving a backend

The backend is chosen from the *type of an example state*. Nothing is configured globally, and
instances are cached, so a rule can call `resolve_backend` per node without paying for it.

In [2]:
print(resolve_backend(np.zeros(4)).name)
print(resolve_backend(torch.zeros(4)).name)
print("cached:", resolve_backend(np.zeros(4)) is resolve_backend(np.ones((3, 8))))

try:
    resolve_backend([0.0, 1.0])            # a list is not an array
except TypeError as exc:
    print("TypeError ->", str(exc).splitlines()[0])

numpy
torch
cached: True
TypeError -> No backend registered for array type <class 'list'>. Subclass specdiff.ops.Backend and pass it to the sampler explicitly.


Inside a `Verifier`, `self.backend_for(request)` does the same thing, which is why no rule in
this library imports a framework:

```python
def verify(self, request):
    ops = self.backend_for(request)                       # numpy or torch, decided by the caller
    noise = ops.randn_stack(1, request.target_mean, request.rng)[0]
    ...
```

## 2. The protocol

These are all the array operations the library depends on.

In [3]:
for name in sorted(Backend.__abstractmethods__):
    doc = (getattr(Backend, name).__doc__ or "-").strip().split("\n")[0]
    print(f"  {name:<20}{doc}")
print()
print("concrete helpers:", [m for m in ("check_state_dtype", "numel") if hasattr(Backend, m)])

  allclose            -
  copy                -
  dot                 Flat inner product of two single states, as a Python float.
  finfo_eps           Machine epsilon of ``ref``'s dtype.
  group_rows          ``(batch * G, *shape) -> (batch, G, *shape)``, preserving order.
  is_finite           -
  is_floating         True if ``ref`` has a floating-point dtype.
  make_rng            A fresh generator this backend's ``randn_stack``/``uniform`` accept.
  norm                Euclidean norm of a single state, as a Python float.
  put                 In-place ``stack[indices] = values``.
  randn_stack         ``(n, *ref.shape)`` standard normal draws.
  repeat_rows         Repeat row ``i`` of ``stack`` ``counts[i]`` times, in order.
  scale_rows          Multiply row ``i`` by ``scalars[i]``.
  stack_rows          Stack single states into ``(len(arrays), *state_shape)``.
  take                Gather rows ``stack[indices]`` (copy).
  uniform             A single scalar draw from Uniform[0, 1

### The four that are not obvious

`repeat_rows` and `group_rows` are what make drafting and verification rectangular;
`scale_rows` is what makes per-row `sigma` portable; `put` is in-place, which is how the samplers
fill their flat node buffers.

In [4]:
ops = resolve_backend(np.zeros(2))
stack = np.arange(6.0).reshape(3, 2)
print("stack\n", stack)

# One parent mean per row, repeated once per child: the drafting step.
print("\nrepeat_rows(stack, [2, 1, 3]) ->\n", ops.repeat_rows(stack, [2, 1, 3]))

# (batch * K, *shape) -> (batch, K, *shape): how a level's candidates become a request.
print("\ngroup_rows(stack, 3).shape ->", ops.group_rows(stack, 3).shape)

# Per-row scaling: sigma differs across rows once trajectories fall out of step.
print("\nscale_rows(stack, [1, 10, 100]) ->\n", ops.scale_rows(stack, [1.0, 10.0, 100.0]))

# put is in-place, take is a copy.
buffer = ops.zeros_stack(4, np.zeros(2))
ops.put(buffer, [0, 3], np.array([[1.0, 1.0], [2.0, 2.0]]))
print("\nbuffer after put([0, 3])\n", buffer)
print("take(buffer, [3, 0])\n", ops.take(buffer, [3, 0]))

stack
 [[0. 1.]
 [2. 3.]
 [4. 5.]]

repeat_rows(stack, [2, 1, 3]) ->
 [[0. 1.]
 [0. 1.]
 [2. 3.]
 [4. 5.]
 [4. 5.]
 [4. 5.]]

group_rows(stack, 3).shape -> (1, 3, 2)

scale_rows(stack, [1, 10, 100]) ->
 [[  0.   1.]
 [ 20.  30.]
 [400. 500.]]

buffer after put([0, 3])
 [[1. 1.]
 [0. 0.]
 [0. 0.]
 [2. 2.]]
take(buffer, [3, 0])
 [[2. 2.]
 [1. 1.]]


The samplers use exactly these operations.

Assume that the diffusion state dimension is `state_shape = (16, 128, 128)` and the number of nodes in the current draft tree is `tree.size = 100`, then `(tree.size, *state_shape) = (100, 16, 128, 128)`. Therefore:
- Scalar `sampler.py` keeps one `(tree.size, *state_shape)` buffer per round and indexes it by node id, with `tree.size` the total number of nodes in the tree (including the root).
- `batched.py` keeps a flat `(rows * tree.size, ...)` buffer indexed by `row * tree.size + node` — which is why the backend needs no gather beyond row indexing.

## 3. Dtypes

States must be floating point. An integer array would truncate every Gaussian draw to zero and
hand back a silently wrong trajectory, so `randn_stack` and `sample()` refuse one outright.

In `ops.check_state_dtype`, `"init"` is the `where` argument, i.e., a purely cosmetic label that gets interpolated into the error message to say which array failed (useful for debugging).

In [5]:
try:
    ops.check_state_dtype(np.zeros(4, dtype=int), "init")
except TypeError as exc:
    print("TypeError ->", str(exc).splitlines()[0])

# Machine epsilon per dtype. Note the degeneracy tolerance is NOT derived from this:
# see DEFAULT_DEGENERATE_TOL. `finfo_eps` is here for rules that want their own.
# in Rank1Frame, and why a fixed constant would be wrong for one of them.
for ref in (np.zeros(4, np.float64), np.zeros(4, np.float32), torch.zeros(4, dtype=torch.float32)):
    b = resolve_backend(ref)
    print(f"{b.name:>6} {str(ref.dtype):<16} eps {b.finfo_eps(ref):.3e}  sqrt(eps) {np.sqrt(b.finfo_eps(ref)):.3e}")

TypeError -> init has non-floating dtype dtype('int64'). Diffusion states must be floating point: an integer array truncates every Gaussian draw to zero and the sampler would return a silently wrong trajectory. Cast with e.g. `init.astype(float)`.
 numpy float64          eps 2.220e-16  sqrt(eps) 1.490e-08
 numpy float32          eps 1.192e-07  sqrt(eps) 3.453e-04
 torch torch.float32    eps 1.192e-07  sqrt(eps) 3.453e-04


`float16` has no guard, and it is a trap: the rank-1 reduction takes a norm and divides by it, so
a small `delta` is pure noise in half precision. Cast to `float32` for the coupling.

## 4. Randomness

`make_rng(seed, ref)` gives a generator the backend's own `randn_stack`/`uniform` accept, without
the caller knowing which framework is in play — that is what lets `check_exactness` seed a run
portably. 

`ref` supplies the device. The reason is torch-specific: a `torch.Generator` is bound to a device, and mixing them is an error. `torch.randn(..., device="cuda, generator=cpu_gen)` raises `Expected a 'cuda' device type for generator but found 'cpu'`. So whoever creates the generator has to know where the states live. For example, `NumpyBackend.make_rng` ignores `ref` entirely, since there's no device to get wrong using numpy.

In [6]:
np_rng = resolve_backend(np.zeros(3)).make_rng(0)
tt_rng = resolve_backend(torch.zeros(3)).make_rng(0, torch.zeros(3))
print(type(np_rng).__name__, "|", type(tt_rng).__name__)

print("numpy draw:", resolve_backend(np.zeros(3)).randn_stack(1, np.zeros(3), np_rng).round(4))
print("torch draw:", resolve_backend(torch.zeros(3)).randn_stack(1, torch.zeros(3), tt_rng))
print("scalar uniform:", round(resolve_backend(np.zeros(3)).uniform(np_rng), 4))
print("scalar uniform:", round(resolve_backend(torch.zeros(3)).uniform(tt_rng), 4))

Generator | Generator
numpy draw: [[ 0.1257 -0.1321  0.6404]]
torch draw: tensor([[ 1.5410, -0.2934, -2.1788]])
scalar uniform: 0.0165
scalar uniform: 0.4556


Two scalar helpers live here too, so that verifiers and tests need no SciPy: $\Phi$ and $\bar{\Phi} = 1 - \Phi$.

In [7]:
print("Phi(0)          ", standard_normal_cdf(0.0))
print("Phi_bar(1.96)   ", round(standard_normal_sf(1.96), 5))
print("eq. (16) alpha at delta=0.5:", round(2.0 * standard_normal_sf(0.5 / 2.0), 4))

Phi(0)           0.5
Phi_bar(1.96)    0.025
eq. (16) alpha at delta=0.5: 0.8026


## 5. The whole sampler on torch

Nothing changes except the arrays. The model returns tensors, the initial state is a tensor, the
generator is a `torch.Generator` — the sampler, the tree and the rule are the same objects.

In [8]:
N, dim = 20, 4
gammas = np.linspace(0.05, 0.25, N)


class TorchLinearKernel(TargetTransition):
    def __init__(self, gammas, dtype=torch.float64):
        super().__init__()
        self.gammas = torch.as_tensor(np.asarray(gammas), dtype=dtype)

    def means(self, indices_in_batch, states, steps):
        gamma = self.gammas[list(steps)].reshape(-1, *([1] * (states.dim() - 1)))
        return states * (1.0 - gamma)


class SqrtSchedule(NoiseSchedule):
    def __init__(self, num_steps, scale=0.4, floor=0.05):
        self.num_steps, self.scale, self.floor = int(num_steps), float(scale), float(floor)

    def sigma(self, step):
        return self.floor + self.scale * float(np.sqrt((self.num_steps - step) / self.num_steps))


class DeltaProbe(Verifier):
    """Framework-agnostic: no numpy, no torch, just ops."""

    name = "delta-probe"

    def __init__(self):
        self.deltas = []

    def reset(self):
        self.deltas.clear()

    def verify(self, request):
        self.deltas.append(Rank1Frame.from_request(request).delta)
        ops = self.backend_for(request)
        noise = ops.randn_stack(1, request.target_mean, request.rng)[0]
        return VerifyResult(request.target_mean + request.sigma * noise, accepted=False)


torch_target = TorchLinearKernel(gammas)
schedule = SqrtSchedule(N)
tree = DraftTree.uniform(branching=2, lookahead=3)

sampler = SpeculativeSampler(
    target=torch_target, proposal=DelayedDriftProposal(torch_target), schedule=schedule,
    tree=tree, verifier=DeltaProbe(), num_steps=N, check_contract=True,
)
gen = torch.Generator().manual_seed(0)
run = sampler.sample(torch.zeros(dim, dtype=torch.float64), rng=gen)

print(run.summary())
print("trajectory:", type(run.trajectory).__name__, tuple(run.trajectory.shape), run.trajectory.dtype)
print("sample:    ", run.sample)

steps=20 target_calls=21 speedup=0.952x acceptance=0.000 drafted=260 verified=131
trajectory: Tensor (21, 4) torch.float64
sample:     tensor([ 0.2806,  0.0129,  0.2197, -0.3291], dtype=torch.float64)


In [9]:
# Batched, on torch, with the native batched delayed drift.
batched = BatchedSpeculativeSampler(
    target=torch_target, proposal=DelayedDriftProposal(torch_target), schedule=schedule,
    tree=tree, verifier=DeltaProbe(), num_steps=N, keep_trajectories=True,
)
init = torch.zeros((6, dim), dtype=torch.float64)
br = batched.sample(init, rng=torch.Generator().manual_seed(0))
print(br.summary())
print("samples:", type(br.samples).__name__, tuple(br.samples.shape))

steps=20 batch=6 target_calls=21 speedup=0.952x (isolated 1.000x, occupancy 1.00) acceptance=0.000 drafted=1560 verified=786
samples: Tensor (6, 4)


In [10]:
# Same accounting as the numpy run of the same configuration -- the backend is an
# implementation detail, not a behaviour change.
class NumpyLinearKernel(TargetTransition):
    def __init__(self, gammas):
        super().__init__()
        self.gammas = np.asarray(gammas, dtype=float)

    def means(self, indices_in_batch, states, steps):
        gamma = self.gammas[list(steps)].reshape(-1, *([1] * (states.ndim - 1)))
        return states * (1.0 - gamma)


np_target = NumpyLinearKernel(gammas)
np_run = SpeculativeSampler(
    target=np_target, proposal=DelayedDriftProposal(np_target), schedule=schedule,
    tree=tree, verifier=DeltaProbe(), num_steps=N,
).sample(np.zeros(dim), rng=np.random.default_rng(0))

print(f"{'':>8}{'NFEs':>6}{'rows':>7}{'drafts':>8}{'speedup':>10}")
for label, r in (("numpy", np_run), ("torch", run)):
    print(f"{label:>8}{r.target_calls:>6}{r.target_states_evaluated:>7}{r.drafted_states:>8}"
          f"{r.speedup:>9.3f}x")

          NFEs   rows  drafts   speedup
   numpy    21    131     260    0.952x
   torch    21    131     260    0.952x


## 6. Testing a torch-native rule

`check_exactness` takes an `array_like` that fixes the backend, so a rule you vectorised in torch
is tested on torch:

In [11]:
from specdiff import ResampleVerifier

for ref, label in ((None, "numpy (default)"),
                   (torch.zeros(8, dtype=torch.float64), "torch float64"),
                   (torch.zeros(8, dtype=torch.float32), "torch float32")):
    report = check_exactness(ResampleVerifier(), delta=1.0, num_children=3, seed=0, array_like=ref)
    print(f"{label:>16}: {report}")

 numpy (default): [PASS] delta=1.000 K=3 n=4000 KS=0.0142 (crit 0.0257) accept=0.000
   torch float64: [PASS] delta=1.000 K=3 n=4000 KS=0.0079 (crit 0.0257) accept=0.000


   torch float32: [PASS] delta=1.000 K=3 n=4000 KS=0.0183 (crit 0.0257) accept=0.000


## 7. Writing a backend

Subclass `Backend`, implement the abstract methods listed in section 2, and hand the instance to
the sampler explicitly — `resolve_backend` only knows about NumPy and torch, and says so:

```python
class MyBackend(Backend):
    name = "mine"
    ...

sampler = SpeculativeSampler(..., backend=MyBackend())
```

The two shipped implementations are the reference, and neither is long:

In [12]:
import inspect

print("NumpyBackend:", len(inspect.getsource(NumpyBackend).splitlines()), "lines")
print("TorchBackend:", len(inspect.getsource(TorchBackend).splitlines()), "lines")
print()
print("passing one explicitly:")
explicit = SpeculativeSampler(
    target=np_target, proposal=DelayedDriftProposal(np_target), schedule=schedule,
    tree=DraftTree.chain(2), verifier=DeltaProbe(), num_steps=5, backend=NumpyBackend(),
)
print(" ", explicit.sample(np.zeros(dim), rng=np.random.default_rng(0)).summary())

NumpyBackend: 75 lines
TorchBackend: 82 lines

passing one explicitly:
  steps=5 target_calls=6 speedup=0.833x acceptance=0.000 drafted=9 verified=10


## Recap

* One module touches arrays; everything else goes through `Backend`.
* States are stacks `(num_nodes, *state_shape)`; ids are plain ints.
* `resolve_backend(example_state)` picks the implementation; `Verifier.backend_for(request)` is
  the in-rule shortcut, which is why no rule imports a framework.
* Float states are enforced. `finfo_eps` exposes per-dtype precision for rules that want a
  dtype-aware tolerance of their own — the *degeneracy* tolerance deliberately is not one.
* The same sampler, tree and rule run on NumPy or torch with identical accounting.

**Next:** [`end_to_end_tutorial.ipynb`](end_to_end_tutorial.ipynb) — all of it on the paper's
Gaussian-mixture setting.